In [3]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config
import ijson
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

client = config.create_minio_client()

/home/panos-varitis/anaconda3/envs/thesis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Bucket('aws'), Bucket('azure'), Bucket('azure-clean'), Bucket('google'), Bucket('google-clean')]


In [4]:
object_name = "AmazonEC2.json"
# object_name = "AmazonTimestream.json"
# object_name = "AmazonS3.json"

TARGET_RECORDS = 100000 
sample_products = []

response = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Το ijson διαβάζει κατευθείαν από το stream byte-byte
    parser = ijson.kvitems(response, 'products')
    
    count = 0
    for sku, product_data in parser:
        #Mε την προοπτική να δημιουργεί πεδίο με sku με την αντίστοιχη τιμή μέσα στο dict αλλά αυτό ήδη υπάρχει
        # product_data['sku'] = sku
        sample_products.append(product_data)
        
        count += 1
        if count >= TARGET_RECORDS:
            break
            
    print(f"Downloaded  {len(sample_products)} records μέσω streaming.")

finally:
    response.close()
    response.release_conn()

# Μετατροπή σε αρχικό DataFrame
df_products = pd.json_normalize(sample_products)
print(f"DataFrame: Rows = {df_products.shape[0]}, Columns = {df_products.shape[1]}")
df_products.head()

Downloaded  100000 records μέσω streaming.
DataFrame: Rows = 100000, Columns = 79


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.currentGeneration,attributes.instanceFamily,attributes.vcpu,attributes.physicalProcessor,attributes.clockSpeed,attributes.memory,attributes.storage,attributes.networkPerformance,attributes.processorArchitecture,attributes.tenancy,attributes.operatingSystem,attributes.licenseModel,attributes.usagetype,attributes.operation,attributes.availabilityzone,attributes.capacitystatus,attributes.classicnetworkingsupport,attributes.dedicatedEbsThroughput,attributes.dedicatedEbsThroughputDescription,attributes.ecu,attributes.enhancedNetworkingSupported,attributes.gpuMemory,attributes.instanceFamilyCategory,attributes.instancesku,attributes.intelAvxAvailable,attributes.intelAvx2Available,attributes.intelTurboAvailable,attributes.marketoption,attributes.normalizationSizeFactor,attributes.preInstalledSw,attributes.regionCode,attributes.servicename,attributes.vpcnetworkingsupport,attributes.processorFeatures,attributes.gpu,attributes.instanceCapacity-12xlarge,attributes.instanceCapacity-16xlarge,attributes.instanceCapacity-24xlarge,attributes.instanceCapacity-2xlarge,attributes.instanceCapacity-32xlarge,attributes.instanceCapacity-4xlarge,attributes.instanceCapacity-8xlarge,attributes.instanceCapacity-Large,attributes.instanceCapacity-Xlarge,attributes.physicalCores,attributes.group,attributes.groupDescription,attributes.transferType,attributes.fromLocation,attributes.fromLocationType,attributes.toLocation,attributes.toLocationType,attributes.fromRegionCode,attributes.toRegionCode,attributes.instanceCapacity-18xlarge,attributes.instanceCapacity-9xlarge,attributes.resourceType,attributes.productType,attributes.provisioned,attributes.volumeApiName,attributes.instanceCapacity-Metal,attributes.instanceCapacity-Medium,attributes.storageMedia,attributes.volumeType,attributes.maxVolumeSize,attributes.maxIopsvolume,attributes.maxIopsBurstPerformance,attributes.maxThroughputvolume,attributes.instance,attributes.snapshotarchivefeetype,attributes.ebsOptimized,attributes.elasticGraphicsType,attributes.instanceCapacity-10xlarge
0,RSH2Y67N4H4CFBQ4,Compute Instance,AmazonEC2,Asia Pacific (Malaysia),AWS Region,c7gd.medium,Yes,Compute optimized,1,AWS Graviton3 Processor,2.5 GHz,2 GiB,1 x 59 NVMe SSD,Up to 12500 Megabit,64-bit,Shared,RHEL,No License required,APS7-Reservation:c7gd.medium,RunInstances:0010,NA,AllocatedCapacityReservation,false,Up to 10000 Mbps,315 Mbps,NA,Yes,NA,Compute Optimized,7CQ8E9D98XGDXC9R,No,No,No,OnDemand,2,NA,ap-southeast-5,Amazon Elastic Compute Cloud,true,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,YHBHTMRVRUX8M33Y,Compute Instance,AmazonEC2,Asia Pacific (Malaysia),AWS Region,m7i.large,Yes,General purpose,2,Intel Xeon Scalable (Sapphire Rapids),3.2 GHz,8 GiB,EBS only,Up to 12500 Megabit,64-bit,Shared,Red Hat Enterprise Linux with HA,No License required,APS7-Reservation:m7i.large,RunInstances:1010,NA,AllocatedCapacityReservation,false,Up to 10000 Mbps,650 Mbps,NA,Yes,NA,General Purpose,SS7SJ8ZYRUB9AY5C,Yes,Yes,Yes,OnDemand,4,NA,ap-southeast-5,Amazon Elastic Compute Cloud,true,Intel AVX; Intel AVX2; Intel AVX512; Intel Tur...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JSEGQBVEG2Y4NYJR,Compute Instance,AmazonEC2,Asia Pacific (Thailand),AWS Region,c6in.24xlarge,Yes,Compute optimized,96,Intel Xeon 8375C (Ice Lake),3.5 GHz,192 GiB,EBS only,150000 Megabit,64-bit,Host,Linux,No License required,APS9-HostBoxUsage:c6in.24xlarge,RunInstances:0100,NA,Used,false,75000 Mbps,75000 Mbps,NA,Yes,NA,Compute Optimized,NaN,Yes,Yes,Yes,OnDemand,192,SQL Ent,ap-southeast-7,Amazon Elastic Compute Cloud,true,Intel AVX; Intel AVX2; Intel AVX512; Intel Turbo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [5]:
# Στο πάνω κελί είχα μία λίστα η οποία είχε μέσα n sku, μαζί με όλα τα attributes τουσ.
# Τώρα στο βήμα αυτό κάνω access την λίστα και απομονώνω σε ένα set μόνο τον κωδικό των n sku (η επιλογή set βασίζεται στην γρήγορη αναζήτηση)
target_skus = {p['sku'] for p in sample_products}

# Αυτή θα είναι η αντίστοιχη sample products του πάνω βήματος. Θα κρατάει τα ζευγάρια sku, με στοιχεία πληρωμής, και μετά θα την κάνουμε dataframe
terms_list = []

parser = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Πάμε βαθύτερα και από το λεξικό terms θα στοχεύσουμε μόνο στις On-Demand υπηρεσίες.
    terms_parser = ijson.kvitems(parser, 'terms.OnDemand')

    # Προφανώς δεν τα θέλω όλα!!Μόνο εκείνα των οποίων το sku βρίσκεται στο set το οποίο δημιούργησα
    for sku, term_offers in terms_parser:
        if sku not in target_skus:
            continue
        
        # Αποθηκεύουμε το sku και ολόκληρο το raw λεξικό των terms του
        terms_list.append({
            'skuNew': sku,
            'termsOndemand': term_offers
        })
        
        # Aπλά για να βεβαιωθώ ότι όσα products πήρα άλλες τόσες και οι τιμές 
        if len(terms_list) >= len(target_skus):
            break
            
    print(f"Downloaded {len(terms_list)} matching terms μέσω streaming.")

finally:
    parser.close()
    parser.release_conn()


df_terms = pd.DataFrame(terms_list)
print(f"Terms DataFrame: Rows = {df_terms.shape[0]}, Columns = {df_terms.shape[1]}")
df_terms.head()

Downloaded 99929 matching terms μέσω streaming.
Terms DataFrame: Rows = 99929, Columns = 2


,skuNew,termsOndemand
0,354NCHJF54DDH7TQ,{'354NCHJF54DDH7TQ.JRTCKXETXF': {'offerTermCod...
1,UM7U8QBMJUK7JDGA,{'UM7U8QBMJUK7JDGA.JRTCKXETXF': {'offerTermCod...
2,NH7HRNPDBQYMGPWJ,{'NH7HRNPDBQYMGPWJ.JRTCKXETXF': {'offerTermCod...
3,4MTNQ5R6GPSE3D5C,{'4MTNQ5R6GPSE3D5C.JRTCKXETXF': {'offerTermCod...
4,JMMKH6VED2E7GWQQ,{'JMMKH6VED2E7GWQQ.JRTCKXETXF': {'offerTermCod...


Θα δουλέψουμε αρχικά με το 2ο dataframe το οποίο περιέχει τα δεδομένα τιμολόγησης. Κρίνονται απαραίτητες 4 ενέργειες
- Άνοιγμα 2η στήλης και άπλωμα δεδομένων
- Αντιστοιχία εσωτερικού sku με αυτό που έβαλα εγώ, και πέταμα μίας στήλης εκ των 2
- Μελέτη για εντοπισμό καθολικών στηλών 
- Αφαίρεση περιττών στηλών
- Κατανόηση pricing και αντιστοίχιση με τους άλλους παρόχους

In [6]:
# Πάιρνω την πρώτη εγγραφή προκειμένου να κάνω έναν έλεγχο των πεδίων
sample_row = df_terms.iloc[0]
print("SKU:", sample_row['skuNew'])

raw_dict = sample_row['termsOndemand']

# Βρίσκουμε το κλειδί (το σύνθετο hash, π.χ. SKU.OfferTermCode)
offer_hash_key = list(raw_dict.keys())[0]
offer_content = raw_dict[offer_hash_key]

print("\n--- Περιεχόμενα προσφοράς (Offer Content) ---")
for k, v in offer_content.items():
    # if k != 'priceDimensions':
    print(f"{k}: {v}")

SKU: 354NCHJF54DDH7TQ

--- Περιεχόμενα προσφοράς (Offer Content) ---
offerTermCode: JRTCKXETXF
sku: 354NCHJF54DDH7TQ
effectiveDate: 2026-06-01T00:00:00Z
priceDimensions: {'354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7': {'rateCode': '354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7', 'description': '$21.254 per On Demand RHEL with SQL Server Enterprise c5.12xlarge Instance Hour', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'Hrs', 'pricePerUnit': {'USD': '21.2540000000'}, 'appliesTo': []}}
termAttributes: {}


Ο παρακάτω κώδικας υλοποιεί ένα μέρος του πρώτου βήματος. Πετάει ένα περιττό hash το οποίο ήταν sku + offerCode, και ανοίγει εν μέρη το λεξικό termsOnDemand το οποίο κατασκεύσαμε όταν πήραμε τα δεδομένα και τα μετατρέψαμε σε datframe. Συγκεκριμένα εξάγει μερικά πεδία και τα κάνει κανονικές στήλες. Το απευθείας normalize δοκιμάστηκε και απετύχε, οπότε και προχωρήσαμε με ένα for loop.

In [7]:
flattened_list = []

for idx, row in df_terms.iterrows():
    skuDefaultValue = row['skuNew']
    raw_dict = row['termsOndemand']
    
    # ΜΠετάω το αρχικό κλειδί το οποίο είχε το dict. Περισσοτερα στην αναφορά
    for hash_key, offer_content in raw_dict.items():
        
        # Εξάγω τα πεδία ένα - ένα και φτιάχνω μία δική μου δομή πιο υύκολη στην ανάλυση
        item = {
            'skuNew': skuDefaultValue,
            'offerTermCode': offer_content.get('offerTermCode'),
            'sku': offer_content.get('sku'),
            'effectiveDate': offer_content.get('effectiveDate'),
            'termAttributes': offer_content.get('termAttributes'),
            'priceDimensions': offer_content.get('priceDimensions') # Το αφήνουμε λεξικό!
        }
        flattened_list.append(item)

df_terms = pd.DataFrame(flattened_list)

print(df_terms.columns)
df_terms.head(2)

Index(['skuNew', 'offerTermCode', 'sku', 'effectiveDate', 'termAttributes',
       'priceDimensions'],
      dtype='object')


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,priceDimensions
0,354NCHJF54DDH7TQ,JRTCKXETXF,354NCHJF54DDH7TQ,2026-06-01T00:00:00Z,{},{'354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7': {'r...
1,UM7U8QBMJUK7JDGA,JRTCKXETXF,UM7U8QBMJUK7JDGA,2026-06-01T00:00:00Z,{},{'UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7': {'r...


Επόμενο βήμα είναι το περαιτέρω άνοιγμα του λεξικού το οποίο κρύβει μέσα τις πληροφοριες τιμολόγησης. Συγκεκριμένα το priceDimensions. Όπως φαίνεται και από το πάνω αποτέλεσμα του κελιού, μέσα στο λεξικό αυτό υπάρχει άλλο ένα περίεργο hash - κλειδί, το οποίο όμως όπως και στο παραπάνω κελί θα απορρίψουμε. Ο κώδικας λοιπόν ακολουθεί την ίδια τακτική: διατρέχει γραμμή γραμμή, προσπερνάει το περίεργο αυτό, και τραβάει μόνο τα πραγματικά δεδομένα.

In [8]:
df_terms.head()

,skuNew,offerTermCode,sku,effectiveDate,termAttributes,priceDimensions
0,354NCHJF54DDH7TQ,JRTCKXETXF,354NCHJF54DDH7TQ,2026-06-01T00:00:00Z,{},{'354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7': {'r...
1,UM7U8QBMJUK7JDGA,JRTCKXETXF,UM7U8QBMJUK7JDGA,2026-06-01T00:00:00Z,{},{'UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7': {'r...
2,NH7HRNPDBQYMGPWJ,JRTCKXETXF,NH7HRNPDBQYMGPWJ,2026-06-01T00:00:00Z,{},{'NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7': {'r...
3,4MTNQ5R6GPSE3D5C,JRTCKXETXF,4MTNQ5R6GPSE3D5C,2026-06-01T00:00:00Z,{},{'4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7': {'r...
4,JMMKH6VED2E7GWQQ,JRTCKXETXF,JMMKH6VED2E7GWQQ,2026-06-01T00:00:00Z,{},{'JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7': {'r...


In [9]:
itemsList = []

for idx, row in df_terms.iterrows():

    # Κατασκευάζουμε εξ'ολοκλήρου νεό dataframe. Δεν κάνουμε ενέργειες πάνω στον υπάρχον. Οπότε φτιάχνουμε γραμμή γραμμή με τα πεδία που έχουμε. Για αυτό και τα εξάξουμε ένα - ένα
    skuNew = row['skuNew']
    offerTermCode = row['offerTermCode']
    skuDefaultValue = row['sku']
    effectiveDate = row['effectiveDate']
    termAttributes = row['termAttributes']
    
    # Παίρνουμε το λεξικό του priceDimensions
    priceDimensionDict = row['priceDimensions']

    # Κοιτάζει εάν όντως το priceDimensions είναι λεξικό. Αν δεν είναι δεν μπαίνει καν μέσα στο loop. Έτσι και δεν κρασάρει, και αποφεύγω να δημιουργήσω γραμμές στις οποίες τα δεδομένα είναι ελλιπή
    if isinstance(priceDimensionDict, dict):
        # ΔΌπως και στο πάνω κελί, με τον τρόπο αυτό αγνοούμε το εσωτετικό xxx.xxx.xxx
        for dimensionsDict_hash_key, dimensionsDict_content in priceDimensionDict.items():
            
            # Το pricePerUnit είναι και αυτό με την σειρά του λεξικού οποτε πριν εφαρμόσω την μέθοδο get προσέχω για να βεβαιωθώ ότι το βρήκα και δεν έπεσα στην περίπτωση "κακών" δεδομένων
            price_per_unit_dict = dimensionsDict_content.get('pricePerUnit', {})
            usd_price = price_per_unit_dict.get('USD') if isinstance(price_per_unit_dict, dict) else None
            
            item = {
                'skuNew': skuNew,
                'offerTermCode': offerTermCode,
                'sku': skuDefaultValue,
                'effectiveDate': effectiveDate,
                'termAttributes': termAttributes,
                'rateCode': dimensionsDict_content.get('rateCode'),
                'description': dimensionsDict_content.get('description'),
                'beginRange': dimensionsDict_content.get('beginRange'),
                'endRange': dimensionsDict_content.get('endRange'),
                'unit': dimensionsDict_content.get('unit'),
                'priceUSD': usd_price,   # Aυτό το πεδίο μέσω του ελέγχου που κάναμε πιο πάνω, ή θα είναι None ή θα έχει κάποια τιμή. Οπότε θα το χρησιμοποιήσω μετά για έλεγχω
                'appliesTo': dimensionsDict_content.get('appliesTo')
            }
            itemsList.append(item)

df_terms_final = pd.DataFrame(itemsList)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (99932, 12)


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,354NCHJF54DDH7TQ,JRTCKXETXF,354NCHJF54DDH7TQ,2026-06-01T00:00:00Z,{},354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Ent...,0,Inf,Hrs,21.2540000000,[]
1,UM7U8QBMJUK7JDGA,JRTCKXETXF,UM7U8QBMJUK7JDGA,2026-06-01T00:00:00Z,{},UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18...,0,Inf,Hrs,12.7440000000,[]
2,NH7HRNPDBQYMGPWJ,JRTCKXETXF,NH7HRNPDBQYMGPWJ,2026-06-01T00:00:00Z,{},NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance...,0,Inf,Hrs,4.2040000000,[]
3,4MTNQ5R6GPSE3D5C,JRTCKXETXF,4MTNQ5R6GPSE3D5C,2026-06-01T00:00:00Z,{},4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c...,0,Inf,Hrs,18.0290000000,[]
4,JMMKH6VED2E7GWQQ,JRTCKXETXF,JMMKH6VED2E7GWQQ,2026-06-01T00:00:00Z,{},JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web with...,0,Inf,Hrs,5.4720000000,[]


##### Επεξεργασία πίνακα με τα δεδομένα τιμολόγησης 
1. Λίστα appliesTo ή οποία μπορεί να φαίνεται κενή, αλλά θα την φροντίσουμε με explode και reindex για κάθε ενδεχόμενο. Επίσης το το πεδίο termAtrributes το οποίο είναι ένα dict κενό, θα το αφαιρέσουμε καθώς μετά από μελέτη του documentaion κρίθηκε άχρηστο αφ'ης στιγμής κρατάμε μόνο On-Demand εγγραφές Δεν χρειάζονται περίεργα loop ή εντολές σύνθετες. Απλή χρήση των 2 εντολών. 

In [10]:
df_terms_final = df_terms_final.explode('appliesTo').reset_index(drop=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (99932, 12)


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,354NCHJF54DDH7TQ,JRTCKXETXF,354NCHJF54DDH7TQ,2026-06-01T00:00:00Z,{},354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Ent...,0,Inf,Hrs,21.2540000000,NaN
1,UM7U8QBMJUK7JDGA,JRTCKXETXF,UM7U8QBMJUK7JDGA,2026-06-01T00:00:00Z,{},UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18...,0,Inf,Hrs,12.7440000000,NaN
2,NH7HRNPDBQYMGPWJ,JRTCKXETXF,NH7HRNPDBQYMGPWJ,2026-06-01T00:00:00Z,{},NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance...,0,Inf,Hrs,4.2040000000,NaN
3,4MTNQ5R6GPSE3D5C,JRTCKXETXF,4MTNQ5R6GPSE3D5C,2026-06-01T00:00:00Z,{},4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c...,0,Inf,Hrs,18.0290000000,NaN
4,JMMKH6VED2E7GWQQ,JRTCKXETXF,JMMKH6VED2E7GWQQ,2026-06-01T00:00:00Z,{},JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web with...,0,Inf,Hrs,5.4720000000,NaN


In [11]:
if 'termAttributes' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['termAttributes'])

print (f"Dimension (rows,cols): {df_terms_final.shape}")
df_terms_final.head()


Dimension (rows,cols): (99932, 11)


,skuNew,offerTermCode,sku,effectiveDate,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,354NCHJF54DDH7TQ,JRTCKXETXF,354NCHJF54DDH7TQ,2026-06-01T00:00:00Z,354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Ent...,0,Inf,Hrs,21.2540000000,NaN
1,UM7U8QBMJUK7JDGA,JRTCKXETXF,UM7U8QBMJUK7JDGA,2026-06-01T00:00:00Z,UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18...,0,Inf,Hrs,12.7440000000,NaN
2,NH7HRNPDBQYMGPWJ,JRTCKXETXF,NH7HRNPDBQYMGPWJ,2026-06-01T00:00:00Z,NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance...,0,Inf,Hrs,4.2040000000,NaN
3,4MTNQ5R6GPSE3D5C,JRTCKXETXF,4MTNQ5R6GPSE3D5C,2026-06-01T00:00:00Z,4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c...,0,Inf,Hrs,18.0290000000,NaN
4,JMMKH6VED2E7GWQQ,JRTCKXETXF,JMMKH6VED2E7GWQQ,2026-06-01T00:00:00Z,JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web with...,0,Inf,Hrs,5.4720000000,NaN


2. Αρχικά παρατηρούμε 2 στήλες με το sku της υπηρεσίας. Η μία ήταν εξαρχής μέσα στα δεδομένα τιμολόγησης με την ονομασία sku, και η άλλη προστέθηκε κατά το άνοιγμα των υπηρεσιών. Συγκεκριμένα όλα τα δεδομένα terms είχαν σαν αρχικό αναγνωριστικό το sku χύμα, και μετά τα δεδομένα: κάπως έτσι "xxx: {dict with pricing info}". Οπότε το αρχικό κλειδί το κάναμε στήλη. Τώρα θα γράψουμε κώδικα ο οποίος ελέγχει αν υπάρχει ταύτιση skuNew με sku, αν δεν υπάρχει θα πετάει την εγγραφή, και στο τέλος θα αφαιρεί μία από τις δύο στήλες.

In [12]:
rowCount = len(df_terms_final)

#Βάζω το if για να μπορώ να τρέχω το κελί και μόνο του χωρίς να πετάει error
if 'skuNew' in df_terms_final.columns and 'sku' in df_terms_final.columns:

    # Φτιάχνω την συνθήκη ελέγχου - διαγραφής μιας υπηρεσίας και την εφαρμόζω απευθείας μετά πάνω στο dataframe. Γλιτώνω το loop 
    condition = (df_terms_final['skuNew'] == df_terms_final['sku'])

    df_filtered_terms = df_terms_final[condition]

    df_terms_final = df_filtered_terms.copy()

print(f"Initial records: {rowCount}")
print(f"Rejected records: {rowCount - len(df_terms_final)}")

if 'skuNew' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['skuNew'])

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Initial records: 99932
Rejected records: 0
Dimensions (rows,cols): (99932, 10)


,offerTermCode,sku,effectiveDate,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,JRTCKXETXF,354NCHJF54DDH7TQ,2026-06-01T00:00:00Z,354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Ent...,0,Inf,Hrs,21.2540000000,NaN
1,JRTCKXETXF,UM7U8QBMJUK7JDGA,2026-06-01T00:00:00Z,UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18...,0,Inf,Hrs,12.7440000000,NaN
2,JRTCKXETXF,NH7HRNPDBQYMGPWJ,2026-06-01T00:00:00Z,NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance...,0,Inf,Hrs,4.2040000000,NaN
3,JRTCKXETXF,4MTNQ5R6GPSE3D5C,2026-06-01T00:00:00Z,4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c...,0,Inf,Hrs,18.0290000000,NaN
4,JRTCKXETXF,JMMKH6VED2E7GWQQ,2026-06-01T00:00:00Z,JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web with...,0,Inf,Hrs,5.4720000000,NaN


3. Η στήλη effectiveDate είναι σε ίδιο μήκος κύματος με τις στήλες που έχει η azure, και η google στα δεδομένα της. Περιγράφει την ημερομηνία και ώρα εκκίνησης της συγκεκριμένης τιμής που υπάρχει για την υπηρεσία. Στην εργασία δεν μας ενδιαφέρει η ιστορικότητα των δεδομέων, οπότε την αφαιρούμε απευθείας με τις αντίστοιχες εντολές.

In [13]:
if 'effectiveDate' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['effectiveDate'])

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (99932, 9)


,offerTermCode,sku,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,JRTCKXETXF,354NCHJF54DDH7TQ,354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Ent...,0,Inf,Hrs,21.2540000000,NaN
1,JRTCKXETXF,UM7U8QBMJUK7JDGA,UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18...,0,Inf,Hrs,12.7440000000,NaN
2,JRTCKXETXF,NH7HRNPDBQYMGPWJ,NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance...,0,Inf,Hrs,4.2040000000,NaN
3,JRTCKXETXF,4MTNQ5R6GPSE3D5C,4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c...,0,Inf,Hrs,18.0290000000,NaN
4,JRTCKXETXF,JMMKH6VED2E7GWQQ,JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web with...,0,Inf,Hrs,5.4720000000,NaN


4. Κλιμακωτές χρεώσεις!! Όλοι οι πάροχοι τις έχουν και σε όλους τις παραλέιπω και κρατάω μόνο την πρώτη βαθμίδα χρέωσης. Το ίδιο και εδω!! Η aws αναπαριστά τις κλιμακωτές χρεώσεις μέσω των πεδίων beginRange και endRange τα οποία καθορίζουν τα όρια. Το 0 στο beginRange είναι αυτό το οποίο μας ενδιαφέρει καθώς είναι η πρώτη - βασική βαθμίδα χρέσωσης. Οπότε θα κρατήσουμε όλες τις εγγραφές εκείνες που έχουν beginRange == 0, και μετά τις στήλες με τα όρια θα τις αφαιρέσουμε, αφού καμία σημασία δεν θα έχουν πλέον.
2 παρατηρήσεις:
- Η aws διατηρεί το ίδιο sku ανάμεσα στις κλίμακες, αλλά αλλάζει το rateCode και συγκεκριμένα τα τελευταία του ψηφία
- Το offerTermCode παρατηρούμε ότι είναι ίδιο για πολλες εγγραφές

In [14]:
tierRates = df_terms_final['beginRange'].unique()
print(tierRates)

['0' '1']


In [15]:
if 'beginRange' in df_terms_final.columns and 'endRange' in df_terms_final.columns:

    df_terms_final = df_terms_final[df_terms_final['beginRange'] == '0']

    df_terms_final.drop(columns=['beginRange', 'endRange'], inplace=True)

df_terms_final.reset_index(drop=True, inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (99929, 7)


,offerTermCode,sku,rateCode,description,unit,priceUSD,appliesTo
0,JRTCKXETXF,354NCHJF54DDH7TQ,354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Ent...,Hrs,21.2540000000,NaN
1,JRTCKXETXF,UM7U8QBMJUK7JDGA,UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18...,Hrs,12.7440000000,NaN
2,JRTCKXETXF,NH7HRNPDBQYMGPWJ,NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance...,Hrs,4.2040000000,NaN
3,JRTCKXETXF,4MTNQ5R6GPSE3D5C,4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c...,Hrs,18.0290000000,NaN
4,JRTCKXETXF,JMMKH6VED2E7GWQQ,JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web with...,Hrs,5.4720000000,NaN


5. Μηδενικές χρεώσεις. Δεν μας ενδιαφέρουν, αποτελούν εξαιρέσεις και ειδικές περιπτώσεις για αυτό και θα τις φιλτράρουμε. Η αιτιολόγηση βρίσκεται στο report και ακολουθεί την ίδια στρατηγική με τους άλλους παρόχους, όπου επίσης απορρίψαμε δωρεάν εγγραφές. Κάνουμε και την στήλη floar για καλύτερες μαθηματικές πράξεις

In [16]:
pd.set_option('display.max_colwidth', None)
df_terms_final['description'].head(5)

0         $21.254 per On Demand RHEL with SQL Server Enterprise c5.12xlarge Instance Hour
1                      $12.744 per On Demand Linux with SQL Std c5.18xlarge Instance Hour
2                                     $4.204 per On Demand SUSE c5.18xlarge Instance Hour
3                  $18.029 per On Demand RHEL with SQL Standard c5.24xlarge Instance Hour
4    $5.472 per On Demand Windows with SQL Web without licenses c5.24xlarge Instance Hour
Name: description, dtype: object

In [17]:
df_terms_final['priceUSD'] = pd.to_numeric(df_terms_final['priceUSD'], errors='coerce')

df_terms_final = df_terms_final[df_terms_final['priceUSD'] > 0]

df_terms_final.reset_index(drop=True, inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (61473, 7)


,offerTermCode,sku,rateCode,description,unit,priceUSD,appliesTo
0,JRTCKXETXF,354NCHJF54DDH7TQ,354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Enterprise c5.12xlarge Instance Hour,Hrs,21.254,NaN
1,JRTCKXETXF,UM7U8QBMJUK7JDGA,UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18xlarge Instance Hour,Hrs,12.744,NaN
2,JRTCKXETXF,NH7HRNPDBQYMGPWJ,NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance Hour,Hrs,4.204,NaN
3,JRTCKXETXF,4MTNQ5R6GPSE3D5C,4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c5.24xlarge Instance Hour,Hrs,18.029,NaN
4,JRTCKXETXF,JMMKH6VED2E7GWQQ,JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web without licenses c5.24xlarge Instance Hour,Hrs,5.472,NaN


5. Σχετικά με την στήλη appliesTo δηλώνει εξάρτηση μεταξ`ύ 2 υπηρεσιών. Δηλαδή εάν η υπηρεσία που βλέπουμε εξαρτάται από κάποια άλλη. Δεν θέλω εξαρτήσεις, άρα υπηρεσίες που δεν έχουν Nan στην στήλη αυτή απορρίπτονται, και κατόπιν απορρίπτεται και η στήλη αυτή καθ'αυτή

In [18]:
apTo = df_terms_final['appliesTo'].unique()
print(apTo)

[nan]


In [19]:
if 'appliesTo' in df_terms_final.columns:
    df_terms_final = df_terms_final[df_terms_final['appliesTo'].isna()]

    df_terms_final.reset_index(drop=True, inplace=True)

    df_terms_final.drop(columns=['appliesTo'], errors='ignore', inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (61473, 6)


,offerTermCode,sku,rateCode,description,unit,priceUSD
0,JRTCKXETXF,354NCHJF54DDH7TQ,354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Enterprise c5.12xlarge Instance Hour,Hrs,21.254
1,JRTCKXETXF,UM7U8QBMJUK7JDGA,UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18xlarge Instance Hour,Hrs,12.744
2,JRTCKXETXF,NH7HRNPDBQYMGPWJ,NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance Hour,Hrs,4.204
3,JRTCKXETXF,4MTNQ5R6GPSE3D5C,4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c5.24xlarge Instance Hour,Hrs,18.029
4,JRTCKXETXF,JMMKH6VED2E7GWQQ,JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web without licenses c5.24xlarge Instance Hour,Hrs,5.472


6. Κοιτάζουμε σε ορισμένες κρίσιμες στήλες, για τιμές με Nan προκειμένου να αφαιρέσουμε τις αντίστοιχες εγγραφές. Με βάση την κοινή λογική θα εξετάσουμε τις στήλες της τιμής, της μονάδας μέτρησης, και του sku, καθώς έαν σε μία από αυτές τις κρίσιμες στήλες απουσιάζει μία τιμή, τα δεδομένα δεν θα μπορούν να αναλυθούν

In [20]:
colsToCheck = ['sku', 'priceUSD', 'unit']
before = len(df_terms_final)

df_terms_final.dropna(subset=colsToCheck, inplace=True)

if before - len(df_terms_final) > 0:
    print(f"Removed {before - len(df_terms_final)} rows due to missing critical values.")

df_terms_final.reset_index(drop=True, inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (61473, 6)


,offerTermCode,sku,rateCode,description,unit,priceUSD
0,JRTCKXETXF,354NCHJF54DDH7TQ,354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Enterprise c5.12xlarge Instance Hour,Hrs,21.254
1,JRTCKXETXF,UM7U8QBMJUK7JDGA,UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18xlarge Instance Hour,Hrs,12.744
2,JRTCKXETXF,NH7HRNPDBQYMGPWJ,NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance Hour,Hrs,4.204
3,JRTCKXETXF,4MTNQ5R6GPSE3D5C,4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c5.24xlarge Instance Hour,Hrs,18.029
4,JRTCKXETXF,JMMKH6VED2E7GWQQ,JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web without licenses c5.24xlarge Instance Hour,Hrs,5.472


7. Η στήλη offerTermCode μετά από εξέταση 5 διαφορετικών υπηρεσιών τιμολόγησης της aws ,φαίνεται να περιέχει τον ίδιο ακριβώς κωδικό. Αυτό διότι το offerTermCode είναι ο παγκόσμιος κωδικός τύπου σύμβασης. Εμείς έχουμε κρατήση σε όλους το On-Demand, οπότε θα είναι πάντα ίδιο. Βάση αυτού η στήλη δεν κρίνεται απαραίτητη στο τελικό dataset και άρα την απορρίπτουμε

In [21]:
df_terms_final['offerTermCode'].unique()

array(['JRTCKXETXF'], dtype=object)

In [22]:
if 'offerTermCode' in df_terms_final.columns:

    df_terms_final.drop(columns=['offerTermCode'], errors='ignore', inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (61473, 5)


,sku,rateCode,description,unit,priceUSD
0,354NCHJF54DDH7TQ,354NCHJF54DDH7TQ.JRTCKXETXF.6YS6EN2CT7,$21.254 per On Demand RHEL with SQL Server Enterprise c5.12xlarge Instance Hour,Hrs,21.254
1,UM7U8QBMJUK7JDGA,UM7U8QBMJUK7JDGA.JRTCKXETXF.6YS6EN2CT7,$12.744 per On Demand Linux with SQL Std c5.18xlarge Instance Hour,Hrs,12.744
2,NH7HRNPDBQYMGPWJ,NH7HRNPDBQYMGPWJ.JRTCKXETXF.6YS6EN2CT7,$4.204 per On Demand SUSE c5.18xlarge Instance Hour,Hrs,4.204
3,4MTNQ5R6GPSE3D5C,4MTNQ5R6GPSE3D5C.JRTCKXETXF.6YS6EN2CT7,$18.029 per On Demand RHEL with SQL Standard c5.24xlarge Instance Hour,Hrs,18.029
4,JMMKH6VED2E7GWQQ,JMMKH6VED2E7GWQQ.JRTCKXETXF.6YS6EN2CT7,$5.472 per On Demand Windows with SQL Web without licenses c5.24xlarge Instance Hour,Hrs,5.472


Τέλος ανάλυσης πίνακα terms!! 
Επειδή το notebook έχει ήδη μεγαλώσει και δεν θέλω η ανάλυση του products να γίνει εδώ αποθηκεύουμε τοπικά στο current directory τα 2 datframe τα οποία επεξεργαζόμαστε (εκείνο με τα προϊόντα και το καθαρισμένο terms), και συνεχίζουμε σε άλλο notebook την ανάλυση του products και το merge

In [ ]:
df_terms_final.to_parquet('cleaned_aws_terms.parquet')
df_products.to_parquet('cleaned_aws_products.parquet')
print("Saved successfully!")

Saved successfully!
